goto commit `6841d90` for this notebook with output.

In [ ]:
%load_ext autoreload
%autoreload 2
import pickle
import numpy as np
import torch
import matplotlib.pyplot as plt

In [ ]:
# Load the NumPy array from a pickle file
with open('../Data/MP_data/displacements0-5000.pkl', 'rb') as file:
    dis_array = pickle.load(file)

In [ ]:
dis_array.shape

In [ ]:
from neuralop.layers.neighbor_search import NeighborSearch
from neuralop.layers.integral_transform import IntegralTransform

In [ ]:
mesh = np.loadtxt('../Data/MP_data/mesh.csv', delimiter=',')

In [ ]:
print(np.max(mesh[0,:]), np.max(mesh[1,:]))

In [ ]:
idx_x = torch.arange(0,2.51+(2.51/100), 2.51/100)
idx_y = torch.arange(0,.42, 0.42/20)
x, y  = torch.meshgrid(idx_x, idx_y, indexing='ij') 


In [ ]:
print(x.shape, y.shape, idx_x.shape, idx_y.shape, torch.max(x), torch.max(y))

In [ ]:
simple_mesh = torch.stack([x.flatten(), y.flatten()]).type(torch.float)

In [ ]:
com_mesh = torch.stack( [torch.tensor(mesh[0,:]),torch.tensor(mesh[1,:])]).type(torch.float)

In [ ]:
print(com_mesh.shape, simple_mesh.shape)

In [ ]:
NS = NeighborSearch(use_open3d=False)
pos_dict_tosimple = NS(torch.transpose(com_mesh, 0, 1), torch.transpose(simple_mesh,0, 1), radius=0.1)

In [ ]:
temp = pos_dict_tosimple['neighbors_row_splits']
temp2 = temp[1:]-temp[:-1]
print(min(temp2),max(temp2))
print(len(temp2))

In [ ]:
print(pos_dict_tosimple.keys())

In [ ]:
NS_2 = NeighborSearch(use_open3d=False)
pos_dict_tocomplex = NS_2( torch.transpose(simple_mesh,0, 1),torch.transpose(com_mesh, 0, 1), radius=0.8)

In [ ]:
temp = pos_dict_tocomplex['neighbors_row_splits']
temp2 = temp[1:]-temp[:-1]
print(min(temp2),max(temp2))
print(len(temp2))

In [ ]:
it = IntegralTransform(mlp_layers=[4,1]).cuda()

In [ ]:
f = com_mesh[0,:]**2+com_mesh[1,:]**2
print(f.shape)

In [ ]:
for key, value in pos_dict_tosimple.items():
    pos_dict_tosimple[key] = pos_dict_tosimple[key].cuda()
with torch.no_grad():
    f_x = it(torch.transpose(com_mesh, 0, 1).cuda(),pos_dict_tosimple, torch.transpose(simple_mesh,0, 1).cuda(),f.reshape(-1,1).cuda() )

In [ ]:
f_x.shape

In [ ]:
plt.scatter(simple_mesh[0,:].cpu(), simple_mesh[1:].cpu(),c=simple_mesh[0,:].cpu()**2 - 3*simple_mesh[1:].cpu()**2 , cmap='viridis', s=1.0)
plt.colorbar()

In [ ]:
torch.transpose(simple_mesh,0, 1).shape

In [ ]:
from data_utils.data_utils import MakserNonuniformMest

In [ ]:
masker = MakserNonuniformMest(torch.transpose(com_mesh, 0, 1).cuda(),torch.transpose(simple_mesh,0, 1).cuda(), 0.04)

In [ ]:
_, mask = masker((1317, 7),max_block=0.7, drop_pix=0.3,\
                 channel_aug_rate = 0.8, channel_drop_rate = 0.4, device='cuda', min_block=10, max_blocks=100 )

In [ ]:
mask[:,1]

In [ ]:
plt.scatter(mesh[0,:], mesh[1:],c=mask[:,4].cpu(), cmap='viridis', s=1.0)
plt.colorbar()

In [ ]:
f

In [ ]:
f[masker.in_nbr[-1].long()==1] = 0

In [ ]:
print(f)

In [ ]:
simple_mesh.shape

In [ ]:
from models.gino import Gino

In [ ]:
model = Gino(1, torch.transpose(com_mesh, 0, 1).cuda(),\
             output_grid=torch.transpose(simple_mesh, 0, 1).cuda(),
             grid_size=(251,42),
             radius=0.08,
             n_modes=[[20,40]]*4,
             scalings=[[1,1]]*4,
             n_heads=[2]*4,
             hidden_token_codim=5,
             gno_mlp_layers=[4,8],
             var_encoding=False,
             var_num=5,
             lifting=False,
             var_enco_channels=2,
             enable_cls_token=False,
              )

In [ ]:
model = model.cuda()
y = model(torch.randn(1, 30, 251, 42).cuda())

In [ ]:
print(y.shape)
#plt.imshow(y[0,1].detach().cpu())

In [ ]:
plt.scatter(mesh[0,:], mesh[1:],c=y[0,:,4].detach().cpu(), cmap='viridis', s=1.0)
plt.colorbar()

In [ ]:
from YParams import YParams
from models.get_models import get_ssl_models_Gino, SslWrapper

In [ ]:
params = YParams('./config/ssl.yaml', 'gino', print_params=True)

In [ ]:
encoder, decoder, contrastive, predictor = get_ssl_models_Gino(params)
model = SslWrapper(params, encoder, decoder, contrastive, predictor, stage='ssl')

In [ ]:
y,_,_,_ = model.cuda()(torch.randn(1,1317,7).cuda())
print(y.shape)

In [ ]:
plt.scatter(mesh[0,:], mesh[1:],c=y[0,:,9].detach().cpu(), cmap='viridis', s=1.0)
plt.colorbar()

In [ ]:
e = encoder.cuda()(torch.randn(1,1317,10).cuda())
print("Encoding ", e.shape)
out = decoder.cuda()(e)
print("Reconstructed", out.shape)
p = predictor.cuda()(e)
print('next time', p.shape)

In [ ]:
from data_utils.data_loaders import get_onestep_dataloader

In [ ]:
train, test = get_onestep_dataloader()

In [ ]:
for i in train:
    print(i[0].shape, i[1].shape)
    #i[0] = torch.nn.functional.normalize(i[0], p=2.0, dim=1)
    for j in range(7):
        print(torch.max(i[0][0,:,j]))
    break

In [ ]:
from train.trainer import simple_trainer

In [ ]:
simple_trainer(model.cuda(), train, test, params)

In [ ]:
model.agumenter_masker.in_nbr.shape

In [ ]:
_, mask = model.agumenter_masker((1317,7))

In [ ]:
plt.scatter(mesh[0,:], mesh[1:],c=mask[:,6].cpu(), cmap='viridis', s=1.0)
plt.colorbar()

In [ ]:
r = torch.randn(5000, 100, 7)
mean = torch.mean(r, dim=(0,1))
var = torch.var(r, dim=(0,1))

In [ ]:
mean

In [ ]:
var

In [ ]:
r = (r - mean)/var

In [ ]:
from models.gnofno import FNOGNO

fnogno = FNOGNO(3,\
            3,\
            torch.transpose(com_mesh, 0, 1),
            torch.transpose(simple_mesh,0, 1),
            ,
)

In [ ]:
k = fnogno(torch.transpose(com_mesh, 0, 1),torch.transpose(com_mesh, 0, 1), torch.randn(1317,3))

In [ ]:
com_mesh.shape